In [ ]:
import os
# python standard library imports
from pathlib import Path
import json
import math
# model building imports
import tensorflow as tf
from keras import Model, layers, Sequential
from keras.applications import EfficientNetV2S, Xception, xception
import keras_cv
# model training imports
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC, F1Score
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
# other imports
from keras.utils import image_dataset_from_directory

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")

# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
tf.config.optimizer.set_jit(True)
print("XLA JIT enabled.")

## Model definitions

In [ ]:
class TransferEfficientNetV2S(Model):
    """
    Pre-trained EfficientNetV2L with RandAugment.
    Note: EfficientNetV2 models include internal rescaling/normalization.
    """

    def __init__(self, num_classes, dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="transfer_effnetv2s")
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate

        # 1. REMOVE: self.rescale_layer (EfficientNet handles this internally)
        
        # 2. UPDATE: Set value_range to (0, 255) because we are passing 
        # raw pixels directly to the augmentation layer now.
        self.augmentation_layer = keras_cv.layers.RandAugment(value_range=(0.0, 255.0))
        
        # 3. FIX: 'classes' argument is only used if include_top=True. 
        # Since we use include_top=False, we leave it out.
        self.base = EfficientNetV2S(
            include_top=False, 
            weights='imagenet' # Ensure weights are loaded
        )
        
        # Freeze the base model if you only want to train the head initially
        self.base.trainable = False 

        self.gap_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(dropout_rate)
        self.dense_layer = layers.Dense(self.num_classes, activation="softmax")

    def unfreeze_base(self, n_freeze=100):
        """
        Phase 2: unfreeze the top layers of the base for fine-tuning.
        n_freeze: number of early layers to keep frozen (they learn generic features
                  that transfer well and don't need retraining).
        """
        self.base.trainable = True
        for layer in self.base.layers[:n_freeze]:
            layer.trainable = False
        frozen = sum(1 for l in self.base.layers if not l.trainable)
        total  = len(self.base.layers)
        print(f"{self.name}: {frozen}/{total} base layers frozen, {total - frozen} unfrozen")

    def get_config(self):
        # Obtém a configuração base da superclasse
        config = super().get_config()
        # Adiciona os teus argumentos personalizados ao dicionário
        config.update({
            "num_classes": self.num_classes,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        # Pass raw [0, 255] pixels to augmentation
        x = self.augmentation_layer(inputs, training=training)
        
        # Pass augmented pixels to EfficientNet (it will rescale them internally)
        x = self.base(x, training=training)
        
        x = self.gap_layer(x)
        return self.dense_layer(x)

In [ ]:
class TransferXception(Model):
    """
    Pre-trained Xception with RandAugment.
    Xception does NOT include internal rescaling — inputs must be in [-1, 1].
    We use xception_preprocess (maps [0,255] -> [-1,1]) AFTER augmentation.
    """

    def __init__(self, num_classes, dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="transfer_xception")
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate

        # Augment on raw [0, 255] pixels before rescaling
        self.augmentation_layer = keras_cv.layers.RandAugment(value_range=(0.0, 255.0))

        self.base = Xception(
            include_top=False,
            weights="imagenet"
        )
        self.base.trainable = False

        self.gap_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(dropout_rate)
        self.dense_layer = layers.Dense(self.num_classes, activation="softmax")

    def unfreeze_base(self, n_freeze=30):
        """
        Phase 2: unfreeze the top layers of the Xception base.
        Xception has ~134 layers — freezing the first 30 preserves low-level features.
        """
        self.base.trainable = True
        for layer in self.base.layers[:n_freeze]:
            layer.trainable = False
        frozen = sum(1 for l in self.base.layers if not l.trainable)
        total  = len(self.base.layers)
        print(f"{self.name}: {frozen}/{total} base layers frozen, {total - frozen} unfrozen")

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_classes": self.num_classes,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        # Step 1: augment on raw uint8 pixels
        x = self.augmentation_layer(inputs, training=training)
        # Step 2: preprocess to [-1, 1] as Xception expects
        x = xception.preprocess_input(x)
        # Step 3: forward through frozen base
        x = self.base(x, training=training)
        x = self.gap_layer(x)
        x = self.dropout_layer(x, training=training)
        return self.dense_layer(x)


## Config and data loading

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
# 384×384: native resolution for EfficientNetV2S (significant accuracy gain over 224)
# Note: ~2.9× more pixels per image — reduce batch_size if you hit OOM on GPU
IMAGE_SIZE     = (384, 384)
BATCH_SIZE     = 16       # reduced from 32 to fit 384px images in 8GB VRAM
PHASE1_EPOCHS  = 15       # frozen-base head training
PHASE2_EPOCHS  = 40       # fine-tuning (EarlyStopping will cut this short)
PHASE1_LR      = 1e-2     # higher LR — only head is updating
PHASE2_LR      = 1e-4     # ~100× lower LR — prevent destroying pretrained weights
N_CLASSES      = 23

data_dir_path = Path("../wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "..\Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "..\Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

## Augmentation and Mixup

In [ ]:
# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
# Forces the model to learn smoother decision boundaries rather than
# memorising exact compositions — especially useful for fine-grained style tasks.
def mixup(images, labels, alpha=0.4):
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

train_ds_mixed = (
    train_ds
    .map(mixup, num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(AUTOTUNE)
)
# val and test are never augmented or mixed
val_ds  = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}


## Model instantiation

In [ ]:

model_effnet   = TransferEfficientNetV2S(num_classes=N_CLASSES)
model_xception = TransferXception(num_classes=N_CLASSES)

transfer_models = [model_effnet, model_xception]

## Metrics and loss

In [ ]:
def make_metrics(num_classes):
    """Return a fresh set of metric instances (metrics are stateful — each model needs its own)."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        F1Score(average="macro", name="f1_score", num_classes=num_classes),
    ]


## Learning rate schedule

In [ ]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Cosine annealing with linear warmup.

    Warmup: LR ramps linearly from 0 to base_lr over the first warmup_epochs.
    This prevents the randomly initialised head from producing large gradients
    that destabilise the pretrained base at the start of training.

    Cosine decay: LR then follows a cosine curve from base_lr down to ~0.
    Finds better minima than step-decay or exponential decay in practice.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler


## Phase 1 — Train heads with frozen base

Only the GAP + Dropout + Dense head is updated.  
The pretrained base is completely frozen.


In [ ]:
phase1_fit_data = {}

for model in transfer_models:
    model_name = model.name
    print(f"\n{'='*60}")
    print(f"Phase 1 training: {model_name}")
    print(f"{'='*60}")


    model.compile(
        optimizer=SGD(learning_rate=PHASE1_LR, decay=1e-4),
        loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
        metrics=make_metrics(num_classes=N_CLASSES),
    )

    callbacks = [
        ModelCheckpoint(
            checkpoints_folder_path / f"ckpt_phase1_{model_name}.tf",
            monitor="val_loss", save_best_only=True, verbose=1,
        ),
        CSVLogger(metrics_folder_path / f"log_phase1_{model_name}.csv"),
        LearningRateScheduler(
            make_cosine_warmup_scheduler(PHASE1_LR, PHASE1_EPOCHS, warmup_epochs=3)
        ),
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ]

    history = model.fit(
        train_ds_mixed,
        validation_data=val_ds,
        epochs=PHASE1_EPOCHS,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )
    phase1_fit_data[model_name] = history

print("\nPhase 1 complete.")


## Phase 2 — Fine-tune unfrozen base layers

Unfreeze the top portion of each pretrained base and retrain at a much lower LR.  
Early layers learn generic features (edges, textures) that transfer well — keep them frozen.  
Later layers learn task-specific patterns — retrain these on art data.


In [ ]:
phase2_fit_data = {}

for model in transfer_models:
    model_name = model.name
    print(f"\n{'='*60}")
    print(f"Phase 2 fine-tuning: {model_name}")
    print(f"{'='*60}")

    # Unfreeze top layers — defaults are set inside each model class
    model.unfreeze_base()

    # Recompile at ~100× lower LR to avoid overwriting pretrained representations
    model.compile(
        optimizer=SGD(learning_rate=PHASE2_LR, decay=1e-5),
        loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
        metrics=make_metrics(num_classes=N_CLASSES),
    )

    callbacks = [
        ModelCheckpoint(
            checkpoints_folder_path / f"ckpt_phase2_{model_name}.tf",
            monitor="val_loss", save_best_only=True, verbose=1,
        ),
        CSVLogger(metrics_folder_path / f"log_phase2_{model_name}.csv"),
        LearningRateScheduler(
            make_cosine_warmup_scheduler(PHASE2_LR, PHASE2_EPOCHS, warmup_epochs=2)
        ),
        # More patience in Phase 2 — improvements are smaller and slower
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
    ]

    history = model.fit(
        train_ds_mixed,
        validation_data=val_ds,
        epochs=PHASE2_EPOCHS,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )
    phase2_fit_data[model_name] = history

print("\nPhase 2 complete.")
